In [ ]:
# Install Pytorch & other libraries, make sure to match your GPU driver version
%pip install --quiet "torch==2.5.1" "setuptools<71.0.0"  --index-url https://download.pytorch.org/whl/cu121
%pip install --quiet flash-attn

%pip install --quiet --upgrade \
  "transformers==4.48.1" \
  "datasets==3.1.0" \
  "accelerate==1.3.0" \
  "hf-transfer==0.1.9" \
  "deepspeed==0.15.4" \
  "trl==0.14.0"

%pip install --quiet "peft==0.14.0" "bitsandbytes==0.45.2"

%pip install --upgrade --quiet sentence_transformers

%pip install --quiet hnswlib
 
%pip install --upgrade ipywidgets

%pip install jinja2==3.1.0

%pip install anthropic

%pip install tf-keras --quiet

%pip install --quiet wandb

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict
from tqdm import tqdm
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from accelerate import Accelerator

HF_HUB_TOKEN = "<INSERT_HF_TOKEN>"

In [4]:
test_file = r'new_test.pkl'

# Load test data
with open(test_file, 'rb') as f:
    test_data = pickle.load(f)
    test_prompts = test_data['prompts']
    test_answers = test_data['answers']

In [ ]:
print(test_prompts[1])
print(test_answers[1])

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.1-8B-Instruct"

# Load the tokenizer and model.
tokenizer = AutoTokenizer.from_pretrained(model_name,
    token=HF_HUB_TOKEN)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(model_name,torch_dtype=torch.float16, token=HF_HUB_TOKEN)

In [10]:
lora_checkpoint = r"checkpoint-1003"

model = PeftModel.from_pretrained(base_model, lora_checkpoint)
model = model.merge_and_unload()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device);

# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = base_model
# model.to(device);

In [11]:
SKILL_DOCSTRINGS = {}
with open('skills/skills_list.pkl', 'rb') as f:
    skill_descs = pickle.load(f)
for s in skill_descs:
    skill = s.split("Skill: ")[1].split("\n\n")[0].strip()
    SKILL_DOCSTRINGS[skill] = s
SKILL_DOCSTRINGS["save_workbook"] = "Helper function to save the active workbook."
SKILL_DOCSTRINGS["paste_special"] = """Perform paste special operation in Excel. If asked to paste special a set of values into a position,
highlight all the values you want to paste at once.

Args:
    start_cell (str): Starting cell reference in A1 notation (e.g., "A1"), denotes the top left cell of copy range.
    end_cell (str): Ending cell reference in A1 notation (e.g., "B2"), denotes the bottom right cell of copy range.
    paste_cell: Target cell for paste (e.g., "C1"), denotes the top left cell of paste range.
    paste_type: Type of paste ("All", "Formulas", "Values", "Formats", "Comments and Notes", "Validation",
                "All using Source theme", "All except borders", "Column widths", "Formula and number formats",
                "Values and number formats", "All, merge conditional formats")
    operation: Math operation ("None", "Add", "Subtract", "Multiply", "Divide")
    skip_blanks: Whether to skip blank cells, is a boolean
    transpose: Whether to transpose the data, is a boolean

Returns:
    Tuple[bool, str]: A tuple containing:
        - bool: True if the operation was successful, False otherwise
        - str: Success/error message from the AppleScript execution
"""

In [12]:
def generate_messages(prompt):
    task = prompt

    intent = task.split("You need to perform the following task for the user:")[-1].split("You have the following skills:")[0].strip()
    if intent.endswith(".."):
        intent = intent[:-1]

    candidate_skills = []
    skills = task.split("```")[1].strip()
    for skill in skills.split("\n"):
        candidate_skills.append(SKILL_DOCSTRINGS[skill.split(":")[0].strip()])
    candidate_skills = "\n".join(candidate_skills)

    messages = [
        {
            "role": "system",
            "content": "You are a computer use agent using Microsoft Excel. You will be given an intent from the user, and you must match it to the most appropriate skill from a list of candidate skills. The skill you choose will then be executed on the computer, and should accomplish the intent.\n\n.You will first reason through your decision and then provide the user with the appropriate skill name. All your thinking should be enclosed in <think></think> tags, you must then provide only the skill name in <answer></answer> tags at the end of your response, e.g. if you decide the appropriate skill is `select_tab` your response will end with <answer>select_tab</answer>."
        },
        {
            "role": "user",
            "content": f"My intent is: {intent}\nSelect a skill from the following list that can accomplish this intent:\n\n{candidate_skills}"
        },
        {
            "role": "assistant",
            "content": "Let me solve this step by step.\n<think>"
        }
    ]

    return messages

In [ ]:
generate_messages(test_prompts[0])

In [ ]:
batch_size = 16

acc = []
output_answers = []
tokenizer.pad_token_id = tokenizer.eos_token_id


# Assume test_prompts is your list of prompts and batch_size is defined.
for i in tqdm(range(0, 1000, batch_size)): # only test on 1000
    batch = test_prompts[i:i+batch_size]
    gt_answers = test_answers[i:i+batch_size]

    batch_conversations = [generate_messages(prompt) for prompt in batch]


    # Tokenize the batch of conversations.
    inputs = tokenizer.apply_chat_template(
        batch_conversations,
        tokenize=True,
        return_tensors="pt",
        add_generation_prompt=False,
        padding=True,
        truncation=True,
        max_length=2048
    ).to(device)

    # Generate text for the batch.
    # Adjust max_new_tokens and other generation parameters as needed.
    output_ids = model.generate(
        inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.2,
        top_p=0.7
    )

    # Decode the generated tokens for the entire batch.
    decoded_outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    # Process each generated output and compare with its corresponding ground truth answer.
    for decoded_output, gt_answer in zip(decoded_outputs, gt_answers):
        # Extract the assistant's answer assuming that the response appears after "assistant"
        output_answer = decoded_output.split("assistant")[-1].strip()
        # print(output_answer)
        substr = "Let me solve this step by step"
        if substr in output_answer:
          output_answer = output_answer.split(substr)[-1]

        if "<answer>" in output_answer and "</answer>" in output_answer:
            output_answer = output_answer.split("<answer>")[1].split("</answer>")[0]
            acc.append(int(gt_answer in output_answer))
        else:
            acc.append(0)

        # print(output_answer)

        output_answers.append(output_answer)

In [ ]:
print(np.mean(acc))